# Laboratory 09 — The fundamental relation

In this laboratory you will hand the computer one function, $S(U, V, N)$, and nothing else — and
get back temperatures, pressures, equilibrium states and heat capacities by differentiating and
maximising it. Then you will run module 01's simulation again and keep its entropy books.

The route: build two fundamental relations and read their slopes (Part 1); find where two solids
stop exchanging energy (Part 2); release a divided gas's wall one constraint at a time (Part 3); check
the slopes against closed forms and measure how the error shrinks (Part 4); keep the entropy
ledger of module 01's simulation (Parts 5 and 6); test extensivity, and break it (Part 7); read
stability off the curvature (Part 8); measure entropy production in a real mixing experiment
(Part 9); run the automated checks (Part 10); and explore freely (Part 11).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a fundamental relation $S(U, V, N)$ — the Einstein solid's $S(U, n)$ and the monatomic ideal gas's Sackur–Tetrode $S(U, V, N)$ — alone or as the two halves of a composite |
| **Dynamics** | none for the surface: releasing a constraint means maximising the total entropy over what the wall lets move; module 01's quantum-hopping simulation supplies one dynamics (Parts 5–6) |
| **Boundary** | the composite is isolated; the internal wall is adiabatic or diathermal, fixed or movable, impermeable or perforated |
| **Ensemble** | equilibrium thermodynamics — every point on the surface is an equilibrium state |
| **Ignored** | fluctuations about the maximum, surface terms, long-range forces |
| **Valid when** | subsystems are macroscopic; the Einstein form where Stirling holds; Sackur–Tetrode for a dilute classical gas |
| **Failure modes** | small systems ($\ln N$ corrections), low temperature (Sackur–Tetrode goes unphysical), convex patches (phase coexistence, long-range forces) |

All the physics lives in `thermolab.fundamental` and `thermolab.equilibrium` — open them and read
them. Nothing in this course is hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import engines, equilibrium, fundamental, processes
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import convergence_study, relative_error, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

QUANTUM = 5.0 * K_B  # the Einstein solids' energy quantum: epsilon / k_B = 5 K
ARGON_MASS = 39.948 * AMU

solid = fundamental.einstein_solid(QUANTUM)
argon = fundamental.monatomic_ideal_gas(ARGON_MASS)
print(f"k_B = {K_B:.6e} J/K,  quantum = {QUANTUM:.3e} J")

## Part 1 — One function, and its slopes

A relation here is just a Python function `s(u, v, n)` returning an entropy in J/K. Nothing else
is supplied: no temperature, no pressure, no equation of state. Take one mole of argon at
$300\ \mathrm{K}$ and $1\ \mathrm{bar}$ — computing $U$ and $V$ for it is the only place we use
what we already know — and ask the function for its slopes.

In [ ]:
n_atoms = N_A
u = 1.5 * n_atoms * K_B * 300.0
v = n_atoms * K_B * 300.0 / 1e5

slopes = fundamental.entropy_slopes(argon, u, v, n_atoms)
print(f"S                  = {float(argon(u, v, n_atoms)):.2f} J/K   "
      f"(tabulated for argon: 154.8 at 298 K)")
print(f"1/T  = dS/dU       -> T  = {slopes.temperature:.6f} K")
print(f"P/T  = dS/dV       -> P  = {slopes.pressure:.4f} Pa")
print(f"-mu/T = dS/dN      -> mu = {slopes.chemical_potential:.4e} J per atom")
print(f"\nPV / (N k_B T)     = {slopes.pressure * v / (n_atoms * K_B * slopes.temperature):.9f}")

### Predict

Commit to an answer for each of the module page's four predictions before going further — they
are what this laboratory tests:

1. A hot 2 kg copper block against a cold 0.2 kg one: equal energies, equal temperatures, both,
   or neither?
2. Can the total entropy of module 01's relaxing solids ever tick *down* for a single step?
3. A gas doubles its volume into a vacuum. Does its entropy rise, fall, or stay fixed?
4. Two identical pieces of a substance whose $S(U)$ curves upwards, in contact at one
   temperature: what happens?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — Where two solids stop

Solid A has 40 oscillators and solid B has 400; together they hold $5 \times 440$ quanta. The wall
between them conducts heat. Sweep every way of sharing the energy, add the two entropies, and
find the peak.

In [ ]:
total = 5.0 * 440 * QUANTUM
pair = fundamental.Composite(solid, solid,
                             fundamental.Part(0.5 * total, 1.0, 40.0),
                             fundamental.Part(0.5 * total, 1.0, 400.0))
u_a, s_a, s_b = pair.energy_scan(n_points=2001, margin=0.002)
s_tot = s_a + s_b
peak = int(np.argmax(s_tot))
share = u_a / total

fig, (top, bottom) = plt.subplots(1, 2, figsize=(11, 3.8))
top.plot(share, (s_tot - s_tot.max()) / K_B, color="black")
top.axvline(share[peak], color="grey", ls=":")
top.set_xlim(0, 0.4)
top.set_ylim(-60, 5)
top.set_xlabel("U_A / U")
top.set_ylabel("(S_A + S_B - S_max) / k_B")
top.set_title("total entropy along the partition")

t_a = [fundamental.temperature_of(solid, x, 1.0, 40.0) for x in u_a[::20]]
t_b = [fundamental.temperature_of(solid, total - x, 1.0, 400.0) for x in u_a[::20]]
bottom.plot(share[::20], t_a, color="#dc2626", label="T_A")
bottom.plot(share[::20], t_b, color="#2563eb", label="T_B")
bottom.axvline(share[peak], color="grey", ls=":")
bottom.set_xlim(0, 0.4)
bottom.set_ylim(0, 150)
bottom.set_xlabel("U_A / U")
bottom.set_ylabel("temperature (K)")
bottom.legend()
bottom.set_title("each solid's temperature, from its own slope")
plt.tight_layout()
plt.show()

best = pair.released("energy")
print(f"scanned peak at U_A/U  = {share[peak]:.4f}")
print(f"maximiser: U_A/U       = {best.a.energy / total:.6f}   "
      f"(n_A / (n_A + n_B) = {40 / 440:.6f})")
print(f"T_A = {best.slopes_a().temperature:.4f} K,   T_B = {best.slopes_b().temperature:.4f} K")
print(f"U_B / U_A              = {best.b.energy / best.a.energy:.4f}")

The peak is where the two temperatures cross, and it sits at one-eleventh of the energy in A.
Equal temperatures, energies in the ratio $1 : 10$ — the equal-energy picture of prediction 1 is
simply not where the maximum is. Notice, too, that the maximiser found the peak without being
told anything about temperature: it only ever compared total entropies.

## Part 3 — Releasing a divided gas's wall, one constraint at a time

A box holds argon split by a wall: side A has a third of the atoms, squeezed into $12\%$ of the
volume, and more than its share of the energy. Release the constraints one at a time and watch
what equalises. Then try the other order.

The laboratory refuses one thing on purpose: freeing the wall to slide while keeping it
insulating. A moving wall does work on both sides, so energy crosses whenever volume does; try
`start.released("volume")` and read the error.

In [ ]:
n_a, n_b = 1e20, 2e20
u_total = 1.5 * (n_a + n_b) * K_B * 300.0
v_total = 3e-3
start = fundamental.Composite(argon, argon,
                              fundamental.Part(0.62 * u_total, 0.12 * v_total, n_a),
                              fundamental.Part(0.38 * u_total, 0.88 * v_total, n_b))


def describe(label, c):
    a, b = c.slopes_a(), c.slopes_b()
    print(f"{label:<26} T = {a.temperature:7.2f} | {b.temperature:7.2f} K   "
          f"P = {a.pressure:9.1f} | {b.pressure:9.1f} Pa   "
          f"mu = {a.chemical_potential:.4e} | {b.chemical_potential:.4e} J   "
          f"S_tot - S_0 = {(c.total_entropy - start.total_entropy) / K_B:.4e} k_B")


describe("as prepared", start)
diathermal = start.released("energy")
describe("wall conducts heat", diathermal)
piston = diathermal.released("energy", "volume")
describe("... and slides", piston)
describe("... and is perforated", piston.released("energy", "particles"))
print()
perforated_first = diathermal.released("energy", "particles")
describe("other order: perforated", perforated_first)
describe("... then slides", perforated_first.released("energy", "volume"))

Each release can only raise the total entropy, and each one equalises exactly one more slope.
Once $T$ and $P$ agree, perforating the wall changes nothing: the Gibbs–Duhem relation leaves
$\mu$ no freedom to differ between two samples of one gas at the same temperature and pressure.
Released in the other order — heat, then holes — the gas ends in the same final state. Look
closely at the entropy column to see why.

## Part 4 — Slopes against closed forms, and how fast the error shrinks

For the Einstein solid the temperature has a closed form,
$T = \varepsilon / (\kB \ln(1 + n\varepsilon/U))$. The laboratory's temperature is a central
difference with a fractional step $h$. A central difference should be second-order: halve $h$
and the error should fall by four.

In [ ]:
u, n = 2.0 * 300 * QUANTUM, 300.0
exact = float(fundamental.einstein_solid_temperature(u, n, QUANTUM))
steps = [5, 10, 20, 40, 80]  # h = 1/steps
study = convergence_study(
    lambda k: fundamental.temperature_of(solid, u, 1.0, n, rel_step=1.0 / k), steps, exact)
for k, err in zip(steps, study.errors, strict=True):
    print(f"h = {1 / k:.4f}   relative error = {err:.3e}")
print(f"\nobserved order = {study.observed_order:.3f}   (central difference: 2)")

print("\nhot solid vs module 01's map T = q epsilon / (n k_B):")
for per_oscillator in (1, 10, 100, 1000):
    u_hot = per_oscillator * n * QUANTUM
    t_exact = float(fundamental.einstein_solid_temperature(u_hot, n, QUANTUM))
    t_equipartition = u_hot / (n * K_B)
    print(f"  {per_oscillator:5d} quanta/oscillator: T = {t_exact:10.3f} K,  "
          f"equipartition {t_equipartition:10.3f} K,  offset {t_exact - t_equipartition:.4f} K")

The order comes out at 2, and as the solid heats the offset from module 01's map settles at
$2.5\ \mathrm{K}$ — half of $\varepsilon/\kB = 5\ \mathrm{K}$. Module 01 assumed the
equipartition map; the counting has derived it, together with its first correction.

## Part 5 — The entropy ledger of module 01's simulation

### Predict

Before running the next cell: module 01's solids — 300 and 100 oscillators, at $500\ \mathrm{K}$
and $250\ \mathrm{K}$ — relax quantum by quantum. You will track their exact total entropy after
every step. Will it ever go down on a single step? Where will it end?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
state = equilibrium.from_temperatures(300, 100, 500.0, 250.0, QUANTUM)
run = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, rng)
ledger = equilibrium.entropy_produced(run) / K_B

total_q = state.total_quanta
q = np.arange(total_q + 1)
exact_count = (equilibrium.einstein_log_multiplicity(q, 300)
               + equilibrium.einstein_log_multiplicity(total_q - q, 100))
ceiling = exact_count.max() - exact_count[state.q_a]

fig, (temps, books) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
temps.plot(run.steps, run.temperature_a, color="#dc2626", lw=0.8, label="T_A")
temps.plot(run.steps, run.temperature_b, color="#2563eb", lw=0.8, label="T_B")
temps.set_ylabel("T (K)")
temps.legend()
books.plot(run.steps, ledger, color="black", lw=0.8)
books.axhline(ceiling, color="grey", ls="--")
books.set_xlabel("step")
books.set_ylabel("Delta S_tot / k_B")
plt.tight_layout()
plt.show()

steps_down = np.diff(ledger) < 0
drawdown = np.max(np.maximum.accumulate(ledger) - ledger)
print(f"final ledger value               = {ledger[-1]:.4f} k_B")
print(f"largest total entropy available  = {ceiling:.4f} k_B")
print(f"fraction of steps that went DOWN = {steps_down.mean():.3f}")
print(f"largest single-step decrease     = {-np.diff(ledger).min():.4f} k_B")
print(f"deepest fall below running best  = {drawdown:.4f} k_B")

Nearly a fifth of all steps lower the total entropy — every time a quantum hops the "wrong" way.
Each dip is tiny, and the ledger still climbs more than a hundred times further than it ever
falls back, flattening just under the largest total entropy any partition of this energy has.
Monotone *within noise* is all a simulation can show. The proof that the flat top is where it
stops is the counting argument of module 08, not this run.

One honest caveat: module 01's hop rule moves quanta as if they were labelled, so its long-run
jitter about the peak is narrower than a true Einstein solid's. It peaks at the same partition,
which is what the ledger tests.

### Three prices for one contact

The same relaxation can be priced with the exact count (the ledger), with the smooth Stirling
surface, and with the calorimetry formula $C_A \ln(T_{\text{eq}}/T_{A,0}) + C_B \ln(T_{\text{eq}}/T_{B,0})$
using module 01's heat capacities $C = n \kB$.

In [ ]:
surface = (fundamental.einstein_solid_entropy(q * QUANTUM, 300, QUANTUM)
           + fundamental.einstein_solid_entropy((total_q - q) * QUANTUM, 100, QUANTUM)) / K_B
closed = fundamental.contact_entropy_production(
    state.heat_capacity_a, state.temperature_a, state.heat_capacity_b, state.temperature_b) / K_B
print(f"exact count (ledger ceiling) = {ceiling:.3f} k_B")
print(f"Stirling surface             = {surface.max() - surface[state.q_a]:.3f} k_B")
print(f"calorimetry formula          = {closed:.3f} k_B")

print("\nshrink the quantum: the Stirling/calorimetry gap is the high-temperature approximation")
for quantum_in_kb in (10.0, 5.0, 2.5, 1.25):
    eps = quantum_in_kb * K_B
    st = equilibrium.from_temperatures(300, 100, 500.0, 250.0, eps)
    qq = np.arange(st.total_quanta + 1)
    surf = (fundamental.einstein_solid_entropy(qq * eps, 300, eps)
            + fundamental.einstein_solid_entropy((st.total_quanta - qq) * eps, 100, eps)) / K_B
    formula = fundamental.contact_entropy_production(
        st.heat_capacity_a, st.temperature_a, st.heat_capacity_b, st.temperature_b) / K_B
    print(f"  epsilon/k_B = {quantum_in_kb:5.2f} K:  surface {surf.max() - surf[st.q_a]:.4f}, "
          f"formula {formula:.4f},  gap {formula - (surf.max() - surf[st.q_a]):.4f}")

## Part 6 — Is the ledger seed-independent?

One run is an anecdote. Repeat it under independent seeds and compare the spread of the final
value with its size.

In [ ]:
def ledger_final(generator):
    trial = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, generator)
    return float(equilibrium.entropy_produced(trial)[-1] / K_B)


def worst_step(generator):
    trial = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, generator)
    return float(-np.diff(equilibrium.entropy_produced(trial)).min() / K_B)


finals = seed_study(ledger_final, n_seeds=8)
dips = seed_study(worst_step, n_seeds=8)
print(f"final value:          {finals.mean:.4f} +/- {finals.standard_error:.4f} k_B   "
      f"(ceiling {ceiling:.4f})")
print(f"worst single step:    {dips.mean:.5f} +/- {dips.standard_error:.5f} k_B")

## Part 7 — Extensivity, three ways

**Free expansion.** Double the volume of 10 000 argon atoms at fixed energy. Compute $\Delta S$
from the Sackur–Tetrode surface, from module 07's ideal-gas formula, and from the heat a
reversible isotherm between the same two states absorbs, divided by its temperature.

**Euler and Gibbs–Duhem.** Check both on the two extensive relations, and on the
self-gravitating "star", whose entropy is not extensive.

In [ ]:
n_small, v1, temperature = 10_000, 1e-20, 300.0
u_small = 1.5 * n_small * K_B * temperature
from_surface = float(argon(u_small, 2 * v1, n_small) - argon(u_small, v1, n_small))
gas = processes.EquilibriumState.from_temperature(n_small, temperature, v1)
from_formula = engines.entropy_change_of_gas(processes.free_expansion(gas, 2 * v1))
from_isotherm = processes.isothermal(gas, 2 * v1).heat / temperature
print(f"N k_B ln 2        = {n_small * K_B * np.log(2):.6e} J/K")
print(f"Sackur-Tetrode    = {from_surface:.6e} J/K")
print(f"module 07 formula = {from_formula:.6e} J/K")
print(f"Q_rev / T         = {from_isotherm:.6e} J/K")

# Flow versus production: the same entropy change reached two ways.
bath = -processes.isothermal(gas, 2 * v1).heat / temperature  # the bath gave that heat up
print("\n                       gas          surroundings   produced   (J/K)")
print(f"reversible isotherm  {from_isotherm:+.3e}   {bath:+.3e}     {from_isotherm + bath:+.1e}")
print(f"free expansion       {from_surface:+.3e}   {0.0:+.3e}     {from_surface:+.3e}")

star = fundamental.self_gravitating_gas(1e-20)
print("\nEuler residual |U - TS + PV - mu N| / |U|:")
print(f"  argon          {fundamental.euler_residual(argon, u, v, n_atoms):.2e}")
print(f"  Einstein solid {fundamental.euler_residual(solid, 20 * 300 * QUANTUM, 1.0, 300.0):.2e}")
print(f"  star           {fundamental.euler_residual(star, -1e-16, 1.0, 1000.0):.6f}")

print("\nGibbs-Duhem residual along a displacement of size d (argon):")
for d in (4e-2, 2e-2, 1e-2, 5e-3):
    r = fundamental.gibbs_duhem_residual(argon, u, v, n_atoms, d * u, 2 * d * v, -d * n_atoms)
    print(f"  d = {d:.4f}:  {r:.3e}")

All three routes to the free expansion agree: the entropy rose by $N \kB \ln 2$ though no heat
flowed in, and the reversible path was only an instrument for computing it (prediction 3). The
little table shows the other half of the story. Along the reversible isotherm the gas gains
exactly what the bath loses: entropy *flows*, and none is produced. In the free expansion the gas
gains the same amount and nothing anywhere loses it: all of it is produced. The
Euler relation balances for both extensive relations and fails on the star by exactly $2|U|$,
which is what a non-extensive entropy predicts — the check can fail, which is what makes it a
check. The Gibbs–Duhem residual falls fourfold each time the displacement halves.

## Part 8 — Stability is curvature

The heat capacity is minus one divided by the product of $T^2$ and the curvature of $S(U)$. Read
it off, then put
two stars in contact and look for the peak of their total entropy.

In [ ]:
c_argon = fundamental.heat_capacity_of(argon, u, v, n_atoms) / (n_atoms * K_B)
c_star = fundamental.heat_capacity_of(star, -1e-16, 1.0, 1000.0) / (1000 * K_B)
print(f"argon  C_V / (N k_B) = {c_argon:.5f}"
      f"   stable: {fundamental.is_stable(argon, u, v, n_atoms)}")
print(f"star   C   / (N k_B) = {c_star:.5f}"
      f"   stable: {fundamental.is_stable(star, -1e-16, 1.0, 1000.0)}")

stars = fundamental.Composite(star, star, fundamental.Part(-1e-16, 1.0, 1000.0),
                              fundamental.Part(-1e-16, 1.0, 1000.0))
u_star, s1, s2 = stars.energy_scan(n_points=401, margin=0.05)
two_solids = fundamental.Composite(solid, solid, fundamental.Part(1e-19, 1.0, 300.0),
                                   fundamental.Part(1e-19, 1.0, 300.0))
u_sol, t1, t2 = two_solids.energy_scan(n_points=401, margin=0.05)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
left.plot(u_sol / two_solids.total("energy"), (t1 + t2 - (t1 + t2).max()) / K_B, color="black")
left.set_title("two identical solids")
right.plot(u_star / stars.total("energy"), (s1 + s2 - (s1 + s2).max()) / K_B, color="#dc2626")
right.set_title("two identical stars")
for ax in (left, right):
    ax.axvline(0.5, color="grey", ls=":")
    ax.set_xlabel("share of the total energy on side A")
    ax.set_ylabel("(S_tot - S_max) / k_B")
plt.tight_layout()
plt.show()

Two identical solids have their maximum at the even split: any fluctuation lowers the total
entropy and is undone. Two identical stars have a *minimum* there: every fluctuation raises the
total entropy, and the pair runs away toward the edges, one star taking the energy (prediction
4). Negative heat capacity and a convex entropy are the same statement.

## Part 9 — Measuring entropy production in a real experiment

`data/09-mixing-calorimetry.csv` is a mixing run: $0.150\ \mathrm{kg}$ of hot water poured into
$0.200\ \mathrm{kg}$ of cold water in a foam cup (read the file's header for exactly what it is).
No single reading is the final temperature — the mixture keeps losing heat to the room — so fit
each trace and extrapolate to the moment of mixing, $t = 0$.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "09-mixing-calorimetry.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "09-mixing-calorimetry.csv")

log = np.genfromtxt(csv_path, delimiter=",", comments="#")
t, t_hot, t_cold, t_mix = log.T
before, after = ~np.isnan(t_hot), (~np.isnan(t_mix)) & (t >= 60)


def extrapolate_to_zero(times, temps):
    # Straight-line fit; the intercept is the temperature at t = 0, with its standard error.
    coeffs, cov = np.polyfit(times, temps, 1, cov=True)
    return coeffs[1], float(np.sqrt(cov[1, 1]))


hot0, d_hot = extrapolate_to_zero(t[before], t_hot[before])
cold0, d_cold = extrapolate_to_zero(t[before], t_cold[before])
final, d_final = extrapolate_to_zero(t[after], t_mix[after])

c_water = 4184.0
c_hot, c_cold = 0.150 * c_water, 0.200 * c_water
predicted_final = (c_hot * hot0 + c_cold * cold0) / (c_hot + c_cold)


def cup_capacity(t_h, t_c, t_f):
    # Energy balance: what the hot water gave up and the cold water did not receive went into
    # the cup and the probe, which started at the cold water's temperature.
    return (c_hot * (t_h - t_f) - c_cold * (t_f - t_c)) / (t_f - t_c)


def produced(t_h, t_c, t_f):
    # Every body that exchanged heat: each changes by C ln(T_final / T_start), module 07's result.
    return (c_hot * np.log(t_f / t_h)
            + (c_cold + cup_capacity(t_h, t_c, t_f)) * np.log(t_f / t_c))


value = produced(hot0, cold0, final)
error = np.sqrt(sum(
    ((produced(hot0 + dh, cold0 + dc, final + df) - value)) ** 2
    for dh, dc, df in ((d_hot, 0, 0), (0, d_cold, 0), (0, 0, d_final))
))

plt.figure(figsize=(8, 3.8))
plt.plot(t[before], t_hot[before], ".", color="#dc2626", label="hot vessel")
plt.plot(t[before], t_cold[before], ".", color="#2563eb", label="cold vessel")
plt.plot(t[~np.isnan(t_mix)], t_mix[~np.isnan(t_mix)], ".", color="black", label="mixture")
plt.axhline(final, color="grey", ls="--", lw=0.8)
plt.axvline(0, color="grey", lw=0.8)
plt.xlabel("t (s)")
plt.ylabel("T (K)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"at t = 0:  hot {hot0:.2f} +/- {d_hot:.2f} K,  cold {cold0:.2f} +/- {d_cold:.2f} K")
print(f"final temperature, extrapolated: {final:.2f} +/- {d_final:.2f} K")
print(f"final temperature, if only the two waters shared the heat: {predicted_final:.2f} K")
print(f"cup and probe, from the energy balance: {cup_capacity(hot0, cold0, final):.1f} J/K")
print(f"\nmeasurement: Delta S_total = {value:.3f} +/- {error:.3f} J/K")
print(f"leaving the cup out of the books:   "
      f"{c_hot * np.log(final / hot0) + c_cold * np.log(final / cold0):.2f} J/K")
print(f"prediction for two waters alone:    "
      f"{fundamental.contact_entropy_production(c_hot, hot0, c_cold, cold0):.2f} J/K")

The entropy produced is positive and far outside its error bar, as it must be. The measured final
temperature sits about $0.4\ \mathrm{K}$ below what the two waters alone would reach: the cup
and the thermometer took some of the heat. The energy balance says how much. A calorimetrist
calls that the cup's *water equivalent*, and it comes out here at about $26\ \mathrm{J\,K^{-1}}$,
roughly six grams of water.

Now look at the line that leaves the cup out. It is badly wrong, and the reason is worth
remembering: the hot water gave up more heat than the cold water received, so books that list
only the two waters do not conserve energy, and an entropy ledger built on them is meaningless.
Entropy production is a total over **every** body that exchanged heat. Including the cup brings
the measurement back in line with the prediction for the two waters.

## Part 10 — Automated checks

A simulation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
# 1. The slopes of Sackur-Tetrode are the ideal-gas equations of state.
assert relative_error(slopes.temperature, 300.0) < 1e-7
assert relative_error(slopes.pressure, 1e5) < 1e-7

# 2. The maximiser lands on equal temperatures and the 1 : 10 energy split (Part 2).
assert relative_error(best.slopes_a().temperature, best.slopes_b().temperature) < 1e-6
assert relative_error(best.b.energy / best.a.energy, 10.0) < 1e-6

# 3. A freed piston equalises the pressures; perforating afterwards changes nothing (Part 3).
assert relative_error(piston.slopes_a().pressure, piston.slopes_b().pressure) < 1e-6
assert relative_error(piston.released("energy", "particles").total_entropy,
                      piston.total_entropy) < 1e-12

# 4. The central difference is second order (Part 4).
assert 1.9 < study.observed_order < 2.1

# 5. The ledger dips on single steps yet ends at the ceiling (Part 5).
assert steps_down.any()
assert ledger[-1] <= ceiling and relative_error(ledger[-1], ceiling) < 0.01

# 6. Free expansion is N k_B ln 2 by all three routes (Part 7).
for route in (from_surface, from_formula, from_isotherm):
    assert relative_error(route, n_small * K_B * np.log(2)) < 1e-9

# 6b. The reversible isotherm moves entropy without producing any (Part 7).
assert abs(from_isotherm + bath) < 1e-9 * from_isotherm

# 7. Euler holds for extensive relations and fails for the star (Part 7).
assert fundamental.euler_residual(argon, u, v, n_atoms) < 1e-6
assert relative_error(fundamental.euler_residual(star, -1e-16, 1.0, 1000.0), 2.0) < 1e-6

# 8. Mixing produces entropy, beyond the measurement error (Part 9).
assert value - 3 * error > 0

print("all checks passed")

## Part 11 — Explore it yourself

Choose the two solids' sizes, the total number of quanta and where the energy starts, then press
**Run Interact**. The left panel shows the total entropy along the partition, with the starting
point and the peak; the right panel runs module 01's simulation from that start and keeps the
ledger. Three things worth trying:

1. Make the solids equal, then make one a hundred times the other. Where is the peak?
2. Start *at* the peak. What does the ledger do?
3. Shrink both solids to a handful of oscillators and watch the ledger's noise grow.

In [ ]:
import ipywidgets as widgets


def explore(n_a=40, n_b=400, quanta=2000, start_share=0.8):
    q_start = int(round(start_share * quanta))
    pair_now = fundamental.Composite(
        solid, solid,
        fundamental.Part(max(q_start, 1) * QUANTUM, 1.0, float(n_a)),
        fundamental.Part(max(quanta - q_start, 1) * QUANTUM, 1.0, float(n_b)))
    ua, sa, sb = pair_now.energy_scan(n_points=801, margin=0.002)
    total_now = pair_now.total("energy")
    peak_now = pair_now.released("energy")

    state_now = equilibrium.TwoBodyState(n_a=n_a, n_b=n_b, q_a=q_start,
                                         q_b=quanta - q_start, quantum=QUANTUM)
    trial = equilibrium.simulate_energy_exchange(state_now, 6 * quanta, rng)
    books_now = equilibrium.entropy_produced(trial) / K_B

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.plot(ua / total_now, (sa + sb - (sa + sb).max()) / K_B, color="black")
    left.axvline(start_share, color="#dc2626", ls="--", label="start")
    left.axvline(peak_now.a.energy / total_now, color="grey", ls=":", label="peak")
    left.set_ylim(max(float(((sa + sb) - (sa + sb).max()).min() / K_B), -200), 5)
    left.set_xlabel("U_A / U")
    left.set_ylabel("(S_tot - S_max) / k_B")
    left.legend()
    right.plot(trial.steps, books_now, color="black", lw=0.8)
    right.set_xlabel("step")
    right.set_ylabel("Delta S_tot / k_B")
    plt.tight_layout()
    plt.show()
    print(f"peak at U_A/U = {peak_now.a.energy / total_now:.4f}   "
          f"T_A = T_B = {peak_now.slopes_a().temperature:.2f} K   "
          f"ledger final = {books_now[-1]:.2f} k_B")


widgets.interact_manual(
    explore,
    n_a=widgets.IntSlider(min=16, max=4096, step=8, value=40, description="n_A"),
    n_b=widgets.IntSlider(min=16, max=4096, step=8, value=400, description="n_B"),
    quanta=widgets.IntSlider(min=100, max=20000, step=100, value=2000, description="quanta"),
    start_share=widgets.FloatSlider(min=0.02, max=0.98, step=0.02, value=0.8,
                                    description="start U_A/U"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "09-fundamental-relation.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. The maximiser in Part 2 never used the word "temperature", yet it landed on equal
   temperatures. Explain why that was inevitable.
3. The ledger in Part 5 went down on thousands of steps. Say precisely what the second law
   does and does not claim about a single step.
4. The star's Euler residual was exactly 2, in units of its energy's size: the Euler relation
   missed twice the energy. What property of its entropy does that number measure?

**Your answers:**

1.
2.
3.
4.